In [5]:

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from pathlib import Path
import time
from pathlib import Path

# Base config
BASE_URL = "https://www.terveyskyla.fi/keuhkotalo/tietoa-keuhkosairauksista/uniapnea"
DOMAIN = "www.terveyskyla.fi"

# Use a relative or absolute path to your sources folder
SOURCES_DIR = Path("sources")
SOURCES_DIR.mkdir(parents=True, exist_ok=True)  # Ensure folder exists

CURRENT_FILE = SOURCES_DIR / "uniapnea_content.txt"
PREVIOUS_FILE = Path("uniapnea_previous_content.txt")

visited = set()
all_text = []

# Set up Selenium
options = Options()
options.headless = True
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

def scrape_page(url):
    print(f"🔍 Scraping: {url}")
    driver.get(url)
    time.sleep(3)

    soup = BeautifulSoup(driver.page_source, "html.parser")

    # Remove structural elements
    for junk in soup.select('[id*="cookie"], [class*="cookie"], footer, nav'):
        junk.decompose()

    lines = []

    for tag in soup.find_all(["h1", "h2", "h3", "p"]):
        text = tag.get_text(strip=True)
        if not text:
            continue

        # Content-based filter to skip cookie/privacy/legal disclaimers
        text_lower = text.lower()
        if any(keyword in text_lower for keyword in [
            "eväste", "cookie", "gdpr", "tietosuoj", "rekisteri", 
            "analytics", "mainonta", "social media", "sivuston käyttö", "some", "hyväksymällä"
        ]):
            continue

        prefix = "# " if tag.name in ["h1", "h2", "h3"] else ""
        lines.append(f"{prefix}{text}")

    if lines:
        all_text.append(f"\n--- {url} ---\n" + "\n".join(lines))

def get_sub_links(base_url):
    driver.get(base_url)
    time.sleep(3)
    soup = BeautifulSoup(driver.page_source, "html.parser")

    links = set()
    for tag in soup.find_all("a", href=True):
        href = tag['href']
        full_url = urljoin(base_url, href)
        parsed = urlparse(full_url)

        if DOMAIN in parsed.netloc and full_url.startswith(BASE_URL):
            links.add(full_url)

    return links

# Step 1: Scrape base page
scrape_page(BASE_URL)
visited.add(BASE_URL)

# Step 2: Crawl sub-pages
sub_links = get_sub_links(BASE_URL)
for link in sub_links:
    if link not in visited:
        scrape_page(link)
        visited.add(link)

driver.quit()

# Step 3: Save new + backup old
if CURRENT_FILE.exists():
    CURRENT_FILE.replace(PREVIOUS_FILE)

with open(CURRENT_FILE, "w", encoding="utf-8") as f:
    f.write("\n\n".join(all_text))

print(f"\n✅ Done! New content saved to '{CURRENT_FILE.name}'")
print(f"📦 Previous version saved as '{PREVIOUS_FILE.name}'")

🔍 Scraping: https://www.terveyskyla.fi/keuhkotalo/tietoa-keuhkosairauksista/uniapnea
🔍 Scraping: https://www.terveyskyla.fi/keuhkotalo/tietoa-keuhkosairauksista/uniapnea/tietoa-uniapneasta
🔍 Scraping: https://www.terveyskyla.fi/keuhkotalo/tietoa-keuhkosairauksista/uniapnea/kokemustarina-uniapnesta
🔍 Scraping: https://www.terveyskyla.fi/keuhkotalo/tietoa-keuhkosairauksista/uniapnea/uniapnea-ja-ajoterveys
🔍 Scraping: https://www.terveyskyla.fi/keuhkotalo/tietoa-keuhkosairauksista/uniapnea/uniapneatutkimukset
🔍 Scraping: https://www.terveyskyla.fi/keuhkotalo/tietoa-keuhkosairauksista/uniapnea#ch2-settings
🔍 Scraping: https://www.terveyskyla.fi/keuhkotalo/tietoa-keuhkosairauksista/uniapnea/uniapnean-itsehoito
🔍 Scraping: https://www.terveyskyla.fi/keuhkotalo/tietoa-keuhkosairauksista/uniapnea#ch2-declaration
🔍 Scraping: https://www.terveyskyla.fi/keuhkotalo/tietoa-keuhkosairauksista/uniapnea#main
🔍 Scraping: https://www.terveyskyla.fi/keuhkotalo/tietoa-keuhkosairauksista/uniapnea/uniapnean

In [7]:

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from pathlib import Path
import json
import time

# Base config
BASE_URL = "https://www.terveyskyla.fi/keuhkotalo/tietoa-keuhkosairauksista/uniapnea"
DOMAIN = "www.terveyskyla.fi"

# Output paths
SOURCES_DIR = Path("sources")
SOURCES_DIR.mkdir(parents=True, exist_ok=True)
CURRENT_JSON = SOURCES_DIR / "uniapnea_chunks.json"
PREVIOUS_JSON = Path("uniapnea_chunks_previous.json")

visited = set()
chunked_data = []

# Set up Selenium
options = Options()
options.headless = True
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

def chunk_text(text, chunk_size=1000, overlap=200):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
    return chunks

def scrape_page(url):
    print(f"🔍 Scraping: {url}")
    driver.get(url)
    time.sleep(3)

    soup = BeautifulSoup(driver.page_source, "html.parser")
    for junk in soup.select('[id*="cookie"], [class*="cookie"], footer, nav'):
        junk.decompose()

    lines = []
    for tag in soup.find_all(["h1", "h2", "h3", "p"]):
        text = tag.get_text(strip=True)
        if not text:
            continue
        text_lower = text.lower()
        if any(keyword in text_lower for keyword in [
            "eväste", "cookie", "gdpr", "tietosuoj", "rekisteri", 
            "analytics", "mainonta", "social media", "sivuston käyttö", "some", "hyväksymällä"
        ]):
            continue
        prefix = "# " if tag.name in ["h1", "h2", "h3"] else ""
        lines.append(f"{prefix}{text}")

    if lines:
        full_text = "\n".join(lines)
        for chunk in chunk_text(full_text):
            chunked_data.append({
                "text": chunk,
                "source": url
            })

def get_sub_links(base_url):
    driver.get(base_url)
    time.sleep(3)
    soup = BeautifulSoup(driver.page_source, "html.parser")

    links = set()
    for tag in soup.find_all("a", href=True):
        href = tag['href']
        full_url = urljoin(base_url, href)
        parsed = urlparse(full_url)
        if DOMAIN in parsed.netloc and full_url.startswith(BASE_URL):
            links.add(full_url)

    return links

# Step 1: Scrape base page
scrape_page(BASE_URL)
visited.add(BASE_URL)

# Step 2: Crawl sub-pages
sub_links = get_sub_links(BASE_URL)
for link in sub_links:
    if link not in visited:
        scrape_page(link)
        visited.add(link)

driver.quit()

# Step 3: Backup previous version
if CURRENT_JSON.exists():
    CURRENT_JSON.replace(PREVIOUS_JSON)

# Step 4: Save to JSON file
with open(CURRENT_JSON, "w", encoding="utf-8") as f:
    json.dump(chunked_data, f, ensure_ascii=False, indent=2)

print(f"\n✅ Done! {len(chunked_data)} chunks saved to '{CURRENT_JSON}'")
print(f"📦 Previous version saved as '{PREVIOUS_JSON}'")

🔍 Scraping: https://www.terveyskyla.fi/keuhkotalo/tietoa-keuhkosairauksista/uniapnea
🔍 Scraping: https://www.terveyskyla.fi/keuhkotalo/tietoa-keuhkosairauksista/uniapnea/tietoa-uniapneasta
🔍 Scraping: https://www.terveyskyla.fi/keuhkotalo/tietoa-keuhkosairauksista/uniapnea/kokemustarina-uniapnesta
🔍 Scraping: https://www.terveyskyla.fi/keuhkotalo/tietoa-keuhkosairauksista/uniapnea/uniapnea-ja-ajoterveys
🔍 Scraping: https://www.terveyskyla.fi/keuhkotalo/tietoa-keuhkosairauksista/uniapnea/uniapneatutkimukset
🔍 Scraping: https://www.terveyskyla.fi/keuhkotalo/tietoa-keuhkosairauksista/uniapnea#ch2-settings
🔍 Scraping: https://www.terveyskyla.fi/keuhkotalo/tietoa-keuhkosairauksista/uniapnea/uniapnean-itsehoito
🔍 Scraping: https://www.terveyskyla.fi/keuhkotalo/tietoa-keuhkosairauksista/uniapnea#ch2-declaration
🔍 Scraping: https://www.terveyskyla.fi/keuhkotalo/tietoa-keuhkosairauksista/uniapnea#main
🔍 Scraping: https://www.terveyskyla.fi/keuhkotalo/tietoa-keuhkosairauksista/uniapnea/uniapnean